# Step 3.4 — Camera-Mono Multi-Frame Tracker (NEW — true monocular baseline) ✅

| | |
|---|---|
| **Input** | `output/step_2/yolo_mono/<sample>/<camera>.json` (Step 2.3.2 — LiDAR-free similar-triangles depth) |
| **Outputs** | `output/step_3/camera_mono/track_<id>.json` — one file per track, trajectory across ALL cameras |
| | `output/step_3/camera_mono_tracking_summary.csv` |
| **Used by** | Step 5 (TTC estimation, true monocular baseline) |

---

Identical tracker design to Step 3.3 (`ClassAwareGlobalTracker`: one Hungarian assignment per frame, class-name gating, `MAX_MISSED_FRAMES` eviction, scene-boundary isolation) — duplicated rather than imported, same rationale CLAUDE.md already documents for `GlobalFrameTracker` (Step 3.1 vs 3.2): the two trackers read different upstream position sources (LiDAR-derived vs. similar-triangles depth) and are expected to diverge in tuning as each baseline matures, so a shared abstraction would need a closer read than a mechanical extraction. The only functional difference from Step 3.3 is the input directory and output track-id prefix (`cammono_` instead of `cam_`, so track ids never collide with Step 3.3's).

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

import shutil
from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import STEP0_DIR, STEP2_DIR, STEP3_DIR

YOLO_MONO_DIR      = STEP2_DIR / "yolo_mono"
CAMERA_MONO_OUT_DIR = STEP3_DIR / "camera_mono"
if CAMERA_MONO_OUT_DIR.exists():
    shutil.rmtree(CAMERA_MONO_OUT_DIR)   # clear stale per-object files before this run (same fix as Steps 3.1-3.3/4)
CAMERA_MONO_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not YOLO_MONO_DIR.exists():
    raise FileNotFoundError(f"Step 2.3.2 output not found at {YOLO_MONO_DIR} — run Step 2.3.2 first.")

print(f"✅ YOLO_MONO_DIR      : {YOLO_MONO_DIR}")
print(f"✅ CAMERA_MONO_OUT_DIR: {CAMERA_MONO_OUT_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ YOLO_MONO_DIR      : F:\Sensor fusion Research\output\step_2\yolo_mono
✅ CAMERA_MONO_OUT_DIR: F:\Sensor fusion Research\output\step_3\camera_mono


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Named constants (same values as Step 3.3)
# ─────────────────────────────────────────────────────────────────

DIST_THRESHOLD      = 2.0   # metres — camera-derived 3D is noisier than LiDAR's own
MAX_MISSED_FRAMES   = 3     # frames

CAMERA_NAMES = ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
                'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']

print(f"✅ DIST_THRESHOLD = {DIST_THRESHOLD}m, MAX_MISSED_FRAMES = {MAX_MISSED_FRAMES}")

✅ DIST_THRESHOLD = 2.0m, MAX_MISSED_FRAMES = 3


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Class-aware global tracker (identical to Step 3.3's)
# ─────────────────────────────────────────────────────────────────

import uuid
import numpy as np
from scipy.optimize import linear_sum_assignment


class ClassAwareGlobalTracker:
    def __init__(self, dist_thresh, max_missed):
        self.active_tracks = {}     # tid -> {"class_name":.., "trajectory":[...], "missed": int}
        self.finished_tracks = {}
        self.dist_thresh = dist_thresh
        self.max_missed = max_missed
        self.current_scene_name = None
        self.association_log = []  # (prev_sample_id, prev_scene, curr_sample_id, curr_scene) per accepted match

    def update(self, detections, sample_id, scene_name):
        """detections: list of dicts with keys pos:[x,y,z], class_name, confidence, camera."""
        if self.current_scene_name is not None and scene_name != self.current_scene_name:
            self.finished_tracks.update(self.active_tracks)
            self.active_tracks = {}
        self.current_scene_name = scene_name

        track_ids = list(self.active_tracks.keys())
        n_tracks, n_dets = len(track_ids), len(detections)

        matched_track_idx, matched_det_idx = set(), set()

        if n_tracks > 0 and n_dets > 0:
            SENTINEL = 1e6
            cost = np.full((n_tracks, n_dets), SENTINEL)

            for i, tid in enumerate(track_ids):
                track = self.active_tracks[tid]
                traj = track["trajectory"]
                last_pos = np.array(traj[-1]["pos"], dtype=float)
                pred = last_pos
                if len(traj) >= 2:
                    vel = (last_pos - np.array(traj[-2]["pos"], dtype=float)) / 0.5   # 2 Hz keyframes
                    spd = np.linalg.norm(vel)
                    if spd > 30.0:
                        vel = vel / spd * 30.0
                    pred = last_pos + vel * 0.5
                for j, det in enumerate(detections):
                    if det["class_name"] != track["class_name"]:
                        continue  # never match across different object classes
                    d = np.linalg.norm(np.array(det["pos"]) - pred)
                    if d < self.dist_thresh:
                        cost[i, j] = d

            row_idx, col_idx = linear_sum_assignment(cost)
            for r, c in zip(row_idx, col_idx):
                if cost[r, c] < SENTINEL:
                    tid = track_ids[r]
                    det = detections[c]
                    prev_sample_id = self.active_tracks[tid]["trajectory"][-1]["sample_id"]
                    self.association_log.append((prev_sample_id, self.current_scene_name, sample_id, scene_name))
                    self.active_tracks[tid]["trajectory"].append({
                        "sample_id": sample_id, "pos": det["pos"],
                        "confidence": det["confidence"], "camera": det["camera"]
                    })
                    self.active_tracks[tid]["missed"] = 0
                    matched_track_idx.add(r)
                    matched_det_idx.add(c)

        for i, tid in enumerate(track_ids):
            if i not in matched_track_idx:
                self.active_tracks[tid]["missed"] += 1
                if self.active_tracks[tid]["missed"] > self.max_missed:
                    self.finished_tracks[tid] = self.active_tracks.pop(tid)

        for j in range(n_dets):
            if j not in matched_det_idx:
                det = detections[j]
                tid = f"cammono_{uuid.uuid4().hex[:8]}"
                self.active_tracks[tid] = {
                    "class_name": det["class_name"],
                    "trajectory": [{
                        "sample_id": sample_id, "pos": det["pos"],
                        "confidence": det["confidence"], "camera": det["camera"]
                    }],
                    "missed": 0
                }

    def save_tracks(self, out_dir):
        all_tracks = {**self.finished_tracks, **self.active_tracks}
        n_saved = 0
        for tid, track in all_tracks.items():
            if len(track["trajectory"]) >= 2:
                with open(out_dir / f"track_{tid}.json", "w") as f:
                    json.dump({
                        "track_id": tid,
                        "class_name": track["class_name"],
                        "trajectory": track["trajectory"]
                    }, f, indent=2)
                n_saved += 1
        return n_saved, len(all_tracks)


print("ClassAwareGlobalTracker defined.")

ClassAwareGlobalTracker defined.


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Main loop: merge all 6 cameras per sample, then track globally
# ─────────────────────────────────────────────────────────────────

import json
from tqdm import tqdm

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

tracker = ClassAwareGlobalTracker(dist_thresh=DIST_THRESHOLD, max_missed=MAX_MISSED_FRAMES)
n_samples_processed = 0
n_detections_skipped_no_3d = 0
n_detections_used = 0

for sample_id in tqdm(sorted(samples_index.keys()), desc="Tracking camera-mono objects"):
    sample_dir = YOLO_MONO_DIR / sample_id
    if not sample_dir.exists():
        continue

    combined_detections = []

    for cam in CAMERA_NAMES:
        cam_file = sample_dir / f"{cam}.json"
        if not cam_file.exists():
            continue

        with open(cam_file) as f:
            dets = json.load(f)

        for det in dets:
            if not det.get("has_3d_position", False):
                n_detections_skipped_no_3d += 1
                continue

            combined_detections.append({
                "pos": [det["global_x"], det["global_y"], det["global_z"]],
                "class_name": det["class_name"],
                "confidence": det["confidence"],
                "camera": cam
            })
            n_detections_used += 1

    scene_name = samples_index[sample_id]["scene_name"]
    tracker.update(combined_detections, sample_id, scene_name)
    n_samples_processed += 1

n_saved, n_total = tracker.save_tracks(CAMERA_MONO_OUT_DIR)

cross_scene_associations = [a for a in tracker.association_log if a[1] != a[3]]
assert len(cross_scene_associations) == 0, \
    f"{len(cross_scene_associations)} cross-scene associations found: {cross_scene_associations[:5]}"

print(f"\nStep 3.4 complete.")
print(f"   Samples processed          : {n_samples_processed}")
print(f"   Detections used (has 3D)   : {n_detections_used}")
print(f"   Detections skipped (no 3D) : {n_detections_skipped_no_3d}")
print(f"   Total tracks created       : {n_total}")
print(f"   Tracks saved (length >= 2)  : {n_saved}")
print(f"   Associations logged        : {len(tracker.association_log)} (0 cross-scene, verified)")
print(f"Saved to: {CAMERA_MONO_OUT_DIR}")

Tracking camera-mono objects:   0%|          | 0/404 [00:00<?, ?it/s]

Tracking camera-mono objects:  16%|█▋        | 66/404 [00:00<00:00, 603.50it/s]

Tracking camera-mono objects:  31%|███▏      | 127/404 [00:00<00:00, 453.21it/s]

Tracking camera-mono objects:  43%|████▎     | 175/404 [00:00<00:00, 380.91it/s]

Tracking camera-mono objects:  63%|██████▎   | 255/404 [00:00<00:00, 505.61it/s]

Tracking camera-mono objects:  77%|███████▋  | 310/404 [00:00<00:00, 448.98it/s]

Tracking camera-mono objects: 100%|██████████| 404/404 [00:00<00:00, 517.59it/s]


Step 3.4 complete.
   Samples processed          : 404
   Detections used (has 3D)   : 6045
   Detections skipped (no 3D) : 0
   Total tracks created       : 2528
   Tracks saved (length >= 2)  : 953
   Associations logged        : 3517 (0 cross-scene, verified)
Saved to: F:\Sensor fusion Research\output\step_3\camera_mono


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Track length distribution + cross-camera continuity check
# ─────────────────────────────────────────────────────────────────

import pandas as pd

track_lengths = []
cross_camera_tracks = 0

for track_file in CAMERA_MONO_OUT_DIR.glob("track_*.json"):
    with open(track_file) as f:
        track = json.load(f)
    track_lengths.append(len(track["trajectory"]))

    cams_seen = {pt["camera"] for pt in track["trajectory"]}
    if len(cams_seen) > 1:
        cross_camera_tracks += 1

length_df = pd.DataFrame({"track_length": track_lengths})
summary_path = STEP3_DIR / "camera_mono_tracking_summary.csv"
length_df.to_csv(summary_path, index=False)

print(f"✅ Summary saved: {summary_path}")
print(f"   Total tracks         : {len(length_df)}")
print(f"   Mean track length    : {length_df['track_length'].mean():.1f} frames")
print(f"   Tracks of length 2   : {(length_df['track_length'] == 2).sum()} "
      f"({(length_df['track_length'] == 2).mean()*100:.1f}%)")
print(f"   Tracks of length 10+ : {(length_df['track_length'] >= 10).sum()}")
print(f"\n   Tracks spanning MORE THAN ONE camera: {cross_camera_tracks} "
      f"({cross_camera_tracks/len(length_df)*100:.1f}% of all tracks)")

display(length_df.describe())

✅ Summary saved: F:\Sensor fusion Research\output\step_3\camera_mono_tracking_summary.csv
   Total tracks         : 953
   Mean track length    : 4.7 frames
   Tracks of length 2   : 367 (38.5%)
   Tracks of length 10+ : 87

   Tracks spanning MORE THAN ONE camera: 458 (48.1% of all tracks)


,track_length
count,953.000000
mean,4.690451
std,4.787466
min,2.000000
25%,2.000000
50%,3.000000
75%,5.000000
max,41.000000
